<a href="https://colab.research.google.com/github/valliansayoga/ey-data-challenge-2025/blob/master/EY2025_Mapping_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install \
    rasterio \
    rioxarray \
    pyproj \
    --quiet --no-cache-dir

In [4]:
!wget https://challenge.ey.com/api/v1/storage/admin-files/556750475068843-678f9918ed9624f63ca75b56-Dataset.zip

--2025-01-28 10:13:28--  https://challenge.ey.com/api/v1/storage/admin-files/556750475068843-678f9918ed9624f63ca75b56-Dataset.zip
Resolving challenge.ey.com (challenge.ey.com)... 52.236.158.32
Connecting to challenge.ey.com (challenge.ey.com)|52.236.158.32|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13861358 (13M) [application/octet-stream]
Saving to: ‘556750475068843-678f9918ed9624f63ca75b56-Dataset.zip.1’

556750475068843-678 100%[===================>]  13.22M  8.16MB/s    in 1.6s    

2025-01-28 10:13:30 (8.16 MB/s) - ‘556750475068843-678f9918ed9624f63ca75b56-Dataset.zip.1’ saved [13861358/13861358]



In [5]:
!unzip 556750475068843-678f9918ed9624f63ca75b56-Dataset.zip
!rm 556750475068843-678f9918ed9624f63ca75b56-Dataset.zip
!unzip TrainTIFF.zip
!rm TrainTIFF.zip
!mv Train TIFF

Archive:  556750475068843-678f9918ed9624f63ca75b56-Dataset.zip
   creating: Dataset/
  inflating: Dataset/2025 EY Open Science AI Data Challenge Participant Guidance.pdf  
  inflating: Dataset/Building_Footprint.kml  
  inflating: Dataset/Landsat_LST.ipynb  
  inflating: Dataset/NY_Mesonet_Weather.xlsx  
  inflating: Dataset/Sentinel2_GeoTIFF.ipynb  
  inflating: Dataset/Submission_template.csv  
  inflating: Dataset/Training_data_uhi_index.csv  
  inflating: Dataset/UHI Experiment Sample Benchmark Notebook V5.ipynb  
Archive:  TrainTIFF.zip
   creating: Train/
  inflating: Train/scl_median.tiff   
  inflating: Train/ndmi_median.tiff  
  inflating: Train/coast_aerosol_median.tiff  
  inflating: Train/ndwi_median.tiff  
  inflating: Train/blue_median.tiff  
  inflating: Train/swir22_median.tiff  
   creating: Train/.ipynb_checkpoints/
  inflating: Train/green_median.tiff  
  inflating: Train/wvp_median.tiff   
  inflating: Train/ndbi_median.tiff  
  inflating: Train/qa_aerosol.tiff   
 

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import rioxarray as rxr
from pyproj import Proj, Transformer, CRS
from tqdm import tqdm
from pathlib import Path
import rioxarray as rxr
from pyproj import Proj, Transformer, CRS
from tqdm import tqdm
from pathlib import Path

tiff_files = list(Path().rglob("*.tiff"))
tiff_files[:5]

[PosixPath('TIFF/wvp_median.tiff'),
 PosixPath('TIFF/coast_aerosol_median.tiff'),
 PosixPath('TIFF/swir16_median.tiff'),
 PosixPath('TIFF/ndbi_median.tiff'),
 PosixPath('TIFF/nir_median.tiff')]

In [16]:
# # Original
# def map_point(row, data):
#     latitudes = row['Latitude']
#     longitudes = row['Longitude']
#     return data.sel(x=longitudes, y=latitudes, band=1, method="nearest").values

# def map_satellite_data(
#     tiff_path,
#     input_df
# ):
#     data = rxr.open_rasterio(tiff_path)
#     tiff_crs = data.rio.crs

#     df = input_df.copy()
#     proj_wgs84 = Proj('epsg:4326')  # EPSG:4326 is the common lat/long CRS
#     proj_tiff = Proj(tiff_crs)

#     # Idk why this is initiated
#     transformer = Transformer.from_proj(proj_wgs84, proj_tiff)

#     # Lightning fast boi
#     values = input_df.apply(map_point, axis=1, data=data)
#     df = pd.DataFrame(values, columns=[tiff_path.stem])
#     return df

def create_csv(tiff_path, output_path, input_df):
    path = Path(output_path)
    path.resolve().parent.mkdir(parents=True, exist_ok=True)
    mapping = map_satellite_data(tiff_path, input_df)
    mapping.to_csv(output_path, index=False)
    return

def create_dataset(df, tiff_files, typ=None, output_path=None):
    for file in tqdm(tiff_files, desc="Creating CSV features..."):
        create_csv(file, f"{typ}/{typ}_{file.stem}.csv", df)

    features = list(Path().resolve().rglob(f"*{typ}_*.csv"))
    features = [pd.read_csv(f) for f in features if "Final" not in f.name]
    df_features = pd.concat([df, *features], axis=1)
    df_features.to_csv(output_path, index=False)
    return df_features

# # # # # # # # With radius!!!
import numpy as np
import pandas as pd
import rioxarray as rxr
from pyproj import Proj, Transformer

def map_point(row, data, radius_pixels):
    """
    Extract data within a specified radius (in pixels) around the nearest lat/long point.
    """
    latitudes = row['Latitude']
    longitudes = row['Longitude']

    # Find the nearest pixel coordinates
    nearest_point = data.sel(x=longitudes, y=latitudes, band=1, method="nearest")
    x_index = np.where(data.x == nearest_point.x)[0][0]
    y_index = np.where(data.y == nearest_point.y)[0][0]

    # Define the window around the nearest point
    x_slice = slice(max(0, x_index - radius_pixels), min(data.x.size, x_index + radius_pixels + 1))
    y_slice = slice(max(0, y_index - radius_pixels), min(data.y.size, y_index + radius_pixels + 1))

    # Extract the data within the window
    window_data = data.isel(x=x_slice, y=y_slice).values

    # Calculate the median of the extracted data (ignoring NaN or no-data values)
    median_value = np.nanmedian(window_data)
    return median_value

def map_satellite_data(
    tiff_path,
    input_df,
    radius_meters=100,
    meter_per_pixel=10
):
    """
    Map satellite data to the input dataframe with a specified radius around each point.
    """
    # Open the TIFF file
    data = rxr.open_rasterio(tiff_path)
    tiff_crs = data.rio.crs

    # Calculate the radius in pixels
    radius_pixels = int(radius_meters / meter_per_pixel)

    # Initialize the coordinate transformer
    proj_wgs84 = Proj('epsg:4326')  # EPSG:4326 is the common lat/long CRS
    proj_tiff = Proj(tiff_crs)
    transformer = Transformer.from_proj(proj_wgs84, proj_tiff)

    # Apply the mapping function to each row in the dataframe
    values = input_df.apply(map_point, axis=1, data=data, radius_pixels=radius_pixels)

    # Create a new dataframe with the results
    df = pd.DataFrame(values, columns=[tiff_path.stem])
    return df

In [ ]:
!rm -r Train
!rm -r Submission

def prepare_train_sub():
    train_path = "/content/Dataset/Training_data_uhi_index.csv"
    sub_path = "/content/Dataset/Submission_template.csv"

    train_df = pd.read_csv(train_path)
    sub_df = pd.read_csv(sub_path)

    typ = "Train"
    out_path = f"{typ}/{typ}_Final.csv"
    print("Creating training data...")
    train_out = create_dataset(train_df, tiff_files=tiff_files, typ=typ, output_path=out_path)
    assert train_df.shape[0] == pd.read_csv(out_path).shape[0], "Column not same between input and output after concatting the train_df!"


    typ = "Submission"
    out_path = f"{typ}/{typ}_Final.csv"
    print("Creating submission data...")
    sub_out = create_dataset(sub_df, tiff_files=tiff_files, typ=typ, output_path=out_path)
    assert sub_df.shape[0] == pd.read_csv(out_path).shape[0], "Column not same between input and output after concatting the sub_df!"

    return train_out, sub_out
train_out, sub_out = prepare_train_sub()

Creating training data...


Creating CSV features...:  28%|██▊       | 5/18 [03:14<08:25, 38.90s/it]

In [13]:
train_out#[["UHI Index", "lwir_median_50m"]]

,Longitude,Latitude,datetime,UHI Index,is_a_building,nearest_building_distance,10m_nearby_building_count,20m_nearby_building_count,30m_nearby_building_count,40m_nearby_building_count,...,qa_aerosol,swir16_median,emsd,lwir_median,swir22_median,scl_median,red_median,ndwi_median,blue_median,ndvi_median
0,-73.909167,40.813107,24-07-2021 15:53,1.030289,0,19.079136,0,1,1,1,...,-0.197360,1980.0,-0.197745,38.393941,1743.0,5.0,0.124005,-0.153698,0.093287,0.216736
1,-73.909187,40.813045,24-07-2021 15:53,1.030289,0,19.233293,0,1,1,1,...,-0.197360,1980.0,-0.197745,38.393941,1743.0,5.0,0.124005,-0.153698,0.093287,0.216736
2,-73.909215,40.812978,24-07-2021 15:53,1.023798,0,20.268009,0,0,1,1,...,-0.197360,1886.0,-0.197745,38.082901,1539.0,5.0,0.097715,-0.199137,0.069170,0.226974
3,-73.909242,40.812908,24-07-2021 15:53,1.023798,0,20.968705,0,0,1,1,...,-0.197360,1823.0,-0.197745,37.785534,1509.0,5.0,0.097715,-0.204715,0.069170,0.241660
4,-73.909257,40.812845,24-07-2021 15:53,1.021634,0,16.324876,0,1,1,2,...,-0.196425,1823.0,-0.197745,37.785534,1509.0,5.0,0.083718,-0.216058,0.062790,0.241660
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11224,-73.957050,40.790333,24-07-2021 15:57,0.972470,0,295.759464,0,0,0,0,...,-0.197360,2002.0,-0.197690,30.440209,1115.0,4.0,0.047500,-0.694981,0.040653,0.780600
11225,-73.957063,40.790308,24-07-2021 15:57,0.972470,0,295.705141,0,0,0,0,...,-0.197360,2007.0,-0.197690,30.440209,1122.0,4.0,0.048765,-0.674734,0.040817,0.769243
11226,-73.957093,40.790270,24-07-2021 15:57,0.981124,0,294.627364,0,0,0,0,...,-0.197360,2007.0,-0.197690,30.440209,1122.0,4.0,0.048765,-0.674734,0.040817,0.769243
11227,-73.957112,40.790253,24-07-2021 15:59,0.981245,0,293.594965,0,0,0,0,...,-0.197360,2007.0,-0.197690,30.440209,1122.0,4.0,0.048765,-0.674734,0.040817,0.769243
